# Notebook recuperado — acervo
Versão sanitizada do Colab encontrado no Drive. Saídas e metadata de sessão removidas. Requer Colab e planilha enviada manualmente. As regras ortográficas são heurísticas históricas não validadas; não é o notebook demonstrativo executado pela CI.


In [ ]:
from google.colab import files
uploaded = files.upload()
input_path = next(iter(uploaded.keys()))
print("Arquivo recebido:", input_path)

In [ ]:
import re
import csv
import unicodedata
from copy import copy
import openpyxl
from openpyxl import Workbook

In [ ]:
def clean_text(s):
    if s is None:
        return ""
    s = str(s)
    s = unicodedata.normalize("NFC", s)
    replacements = {
        "’": "'", "‘": "'", "´": "'", "`": "'", "ʹ":"'", "′":"'", "ʻ":"'",
        "–":"-", "—":"-", "‑":"-", "‐":"-", "−":"-",
        "“":'"', "”":'"',
        "…":"..."
    }
    for a, b in replacements.items():
        s = s.replace(a, b)
    s = re.sub(r"\s+", " ", s).strip()
    s = re.sub(r"-{2,}", "-", s)
    s = re.sub(r"\s*-\s*", "-", s)
    s = re.sub(r"\s*'\s*", "'", s)
    return s

def convert_j_to_circumflex_i(s):
    return s.replace("J", "Î").replace("j", "î")

def replace_qc_to_ks(s):
    s = re.sub(r"QÜ", "KÛ", s)
    s = re.sub(r"GÜ", "GÛ", s)
    s = re.sub(r"qü", "kû", s)
    s = re.sub(r"gü", "gû", s)
    s = re.sub(r"QU", "K", s)
    s = re.sub(r"qu", "k", s)
    s = re.sub(r"C(?=[EIYÉÍÝÊĨỸ])", "S", s)
    s = re.sub(r"c(?=[eiyéíýêĩỹ])", "s", s)
    s = re.sub(r"C", "K", s)
    s = re.sub(r"c", "k", s)
    return s

def replace_s_variants(s):
    s = s.replace("Ç", "S").replace("ç", "s")
    s = re.sub(r"SS", "S", s)
    s = re.sub(r"ss", "s", s)
    return s

def convert_specific_gu_semivowels(s):
    patterns = [
        (r"(^|[-'])GUA", r"\1GÛA"),
        (r"(^|[-'])GUE", r"\1GÛE"),
        (r"(^|[-'])GUI", r"\1GÛI"),
        (r"(^|[-'])GUY", r"\1GÛY"),
        (r"(^|[-'])GOA", r"\1GÛA"),
        (r"(^|[-'])GOE", r"\1GÛE"),
        (r"(^|[-'])GOI", r"\1GÛI"),
        (r"(^|[-'])GOY", r"\1GÛY"),
        (r"(^|[-'])KUA", r"\1KÛA"),
        (r"(^|[-'])KUE", r"\1KÛE"),
        (r"(^|[-'])KUI", r"\1KÛI"),
        (r"(^|[-'])KUY", r"\1KÛY"),
    ]
    for pat, rep in patterns:
        s = re.sub(pat, rep, s)
    return s

def convert_common_suffixes(s):
    s = s.replace("MIRI", "MIRĨ")
    return s

term_overrides = {
    "AI": "AÎ",
    "AIBA": "AÍBA",
    "AIB-ARA": "AÍB-ARA",
    "AIB-E": "AÍB-E",
    "AIB'-I": "AÍB'-I",
    "AIA": "ÃÎA",
    "AIA-PUÃ": "ÃÎA-PUÃ",
    "AI-BARA": "ÃÎ-BARA",
    "AI-BA-Ú": "ÃÎ-BA-Ú",
    "AI-BORA": "ÃÎ-BORA",
    "AI-ÇABA": "AÎ-SABA",
    "AI-ÇY-YRA": "AÎ-SY-YRA",
    "AI-GOERA": "AÎ-GÛERA",
    "AI-IOARA": "ÃÎ-IOARA",
    "AIPI": "AÎPI",
    "AIPI-MAKASERA": "AÎPI-MAKASERA",
    "AIPI-MIXIRA": "AÎPI-MIXIRA",
    "AIPI-Y'": "AÎPI-Y'",
    "AIPÓ-BAE": "AÎPÓ-BA'E",
    "AIPÓ-ÇUI": "AÎPÓ-SUÎ",
    "AIR-A": "AÎR-A",
    "AIURU-ÍLUBA-KANGA": "AÎURU-ÎUBA-KANGA",
    "AIXO": "AÎXÓ",
    "AIÍXBA": "AÎYBA",
    "AIVY-BENA": "AÎY-BENA",
    "AMYIA": "AMYÎA",
    "AMYI-'OKA": "AMYÎ-'OKA",
    "AMYNI U": "AMYNÎÎU",
    "AMYNI-TU-YBA": "AMYNÎÎU-YBA",
    "AKAIU-TEBIRÔ": "AKAÎU-TEBIRÕ",
    "Á": "Ã",
    "Á XE": "Ã XE",
    "AÂ-NDE": "AÃ-NDE",
    "AAN E-NHÊ": "AAN-E-NHÊ",
}

def normalize_navarro(s):
    s0 = clean_text(s).upper()
    if not s0:
        return ""
    if s0 in term_overrides:
        return term_overrides[s0]
    s1 = s0
    s1 = replace_s_variants(s1)
    s1 = convert_j_to_circumflex_i(s1)
    s1 = replace_qc_to_ks(s1)
    s1 = convert_specific_gu_semivowels(s1)
    s1 = convert_common_suffixes(s1)
    replacements = {
        "GUAÇU": "GÛASU",
        "GOAÇU": "GÛASU",
        "GUASU": "GÛASU",
        "GOASU": "GÛASU",
        "GUYRA": "GÛYRA",
        "GOYRA": "GÛYRA",
        "GUARA": "GÛARA",
        "GOARA": "GÛARA",
        "GUERA": "GÛERA",
        "GOERA": "GÛERA",
        "GUATA": "GÛATA",
        "GOATA": "GÛATA",
    }
    for a, b in replacements.items():
        s1 = s1.replace(a, b)
    s1 = re.sub(r"-{2,}", "-", s1)
    s1 = re.sub(r"\s+", " ", s1).strip()
    return s1

In [ ]:
wb_src = openpyxl.load_workbook(input_path)
ws_src = wb_src[wb_src.sheetnames[0]]

header_style = {k: copy(getattr(ws_src.cell(1, 1), k)) for k in ['font', 'fill', 'alignment', 'border', 'protection', 'number_format']}
even_style = {k: copy(getattr(ws_src.cell(2, 1), k)) for k in ['font', 'fill', 'alignment', 'border', 'protection', 'number_format']}
odd_style = {k: copy(getattr(ws_src.cell(3, 1), k)) for k in ['font', 'fill', 'alignment', 'border', 'protection', 'number_format']}

wb_new = Workbook()
ws_new = wb_new.active
ws_new.title = ws_src.title

headers = ["TUPI O.O.", "TUPI O.N.", "PTBR", "ENUS", "EXPLICAÇÃO PTBR"]

for c, h in enumerate(headers, start=1):
    cell = ws_new.cell(1, c, h)
    for attr, val in header_style.items():
        setattr(cell, attr, copy(val))

ws_new.row_dimensions[1].height = ws_src.row_dimensions[1].height

widths = {
    'A': ws_src.column_dimensions['A'].width or 24,
    'B': ws_src.column_dimensions['A'].width or 24,
    'C': ws_src.column_dimensions['B'].width or 34,
    'D': ws_src.column_dimensions['D'].width or 34,
    'E': ws_src.column_dimensions['C'].width or 85,
}

for col, width in widths.items():
    ws_new.column_dimensions[col].width = width

for src_r in range(2, ws_src.max_row + 1):
    tupi_oo = ws_src.cell(src_r, 1).value
    ptbr = ws_src.cell(src_r, 2).value
    explicacao = ws_src.cell(src_r, 3).value
    enus = ws_src.cell(src_r, 4).value
    tupi_on = normalize_navarro(tupi_oo)

    row_values = [tupi_oo, tupi_on, ptbr, enus, explicacao]
    style = even_style if src_r % 2 == 0 else odd_style

    for c, value in enumerate(row_values, start=1):
        cell = ws_new.cell(src_r, c, value)
        for attr, val in style.items():
            setattr(cell, attr, copy(val))

ws_new.freeze_panes = "A2"
ws_new.auto_filter.ref = f"A1:E{ws_new.max_row}"

output_xlsx = "tupi_carvalho_navarro_tabela.xlsx"
output_csv = "tupi_carvalho_navarro_tabela.csv"

wb_new.save(output_xlsx)

with open(output_csv, "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.writer(f)
    writer.writerow(headers)
    for r in range(2, ws_new.max_row + 1):
        writer.writerow([ws_new.cell(r, c).value for c in range(1, 6)])

print("Arquivos gerados:", output_xlsx, output_csv)

In [ ]:
for r in [2, 3, 240, 245, 285, 323, 330, 357, 577, 582]:
    print(r, [ws_new.cell(r, c).value for c in range(1, 6)])

In [ ]:
files.download(output_xlsx)
files.download(output_csv)